In [1]:
import os
import cv2
import ctypes
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

# =========================
# LOAD RESIZE LIBRARY
# =========================
lib_resize = ctypes.CDLL("./libresize.so")

lib_resize.resize_image.argtypes = [
    ctypes.POINTER(ctypes.c_double),  # input
    ctypes.c_int,                     # old_h
    ctypes.c_int,                     # old_w
    ctypes.POINTER(ctypes.c_double),  # output
    ctypes.c_int,                     # new_h
    ctypes.c_int                      # new_w
]

lib_resize.resize_image.restype = None

# =========================
# LOAD FASHION-MNIST
# =========================
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

print("Train:", x_train.shape)
print("Test :", x_test.shape)

# =========================
# OUTPUT DIRECTORY
# =========================
base_dir = "/home/ubuntu/fmnist_images_resized"

# =========================
# CREATE FOLDER STRUCTURE
# =========================
for split in ["train", "test"]:
    for label in range(10):
        os.makedirs(
            os.path.join(base_dir, split, str(label)),
            exist_ok=True
        )

print("Folders created!")

# =========================
# RESIZE FUNCTION
# 28x28 grayscale
#      ↓
# RGB (3 channels)
#      ↓
# libresize.so
#      ↓
# 64x64 RGB
# =========================
def resize_fmnist_lib(img):

    # Grayscale -> RGB
    img_rgb = np.stack([img] * 3, axis=-1).astype(np.float64)

    old_h, old_w = img_rgb.shape[:2]

    out = np.zeros((64, 64, 3), dtype=np.float64)

    lib_resize.resize_image(
        img_rgb.ravel().ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        old_h,
        old_w,
        out.ravel().ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        64,
        64
    )

    return np.clip(out, 0, 255).astype(np.uint8)

# =========================
# SAVE TRAIN IMAGES
# =========================
for idx, (img, label) in enumerate(zip(x_train, y_train)):

    img_resized = resize_fmnist_lib(img)

    save_path = os.path.join(
        base_dir,
        "train",
        str(label),
        f"{idx}.png"
    )

    cv2.imwrite(save_path, img_resized)

    if (idx + 1) % 5000 == 0:
        print(f"Train: {idx+1}/{len(x_train)} saved")

# =========================
# SAVE TEST IMAGES
# =========================
for idx, (img, label) in enumerate(zip(x_test, y_test)):

    img_resized = resize_fmnist_lib(img)

    save_path = os.path.join(
        base_dir,
        "test",
        str(label),
        f"{idx}.png"
    )

    cv2.imwrite(save_path, img_resized)

    if (idx + 1) % 2000 == 0:
        print(f"Test: {idx+1}/{len(x_test)} saved")

print("✅ Fashion-MNIST resized to 64×64 using libresize.so and saved successfully!")

# =========================
# VERIFY ONE IMAGE
# =========================
sample_path = os.path.join(base_dir, "train", "0")

files = sorted(os.listdir(sample_path))
if files:
    sample = cv2.imread(
        os.path.join(sample_path, files[0]),
        cv2.IMREAD_UNCHANGED
    )

    print("\nVerification:")
    print("Shape:", sample.shape)
    print("Dtype:", sample.dtype)

    if len(sample.shape) == 3:
        print("Channels:", sample.shape[2])
        print("R==G:", np.all(sample[:,:,0] == sample[:,:,1]))
        print("G==B:", np.all(sample[:,:,1] == sample[:,:,2]))

I0000 00:00:1781973322.490169 1862466 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781973322.936408 1862466 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781973324.572578 1862466 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Train: (60000, 28, 28)
Test : (10000, 28, 28)
Folders created!
Train: 5000/60000 saved
Train: 10000/60000 saved
Train: 15000/60000 saved
Train: 20000/60000 saved
Train: 25000/60000 saved
Train: 30000/60000 saved
Train: 35000/60000 saved
Train: 40000/60000 saved
Train: 45000/60000 saved
Train: 50000/60000 saved
Train: 55000/60000 saved
Train: 60000/60000 saved
Test: 2000/10000 saved
Test: 4000/10000 saved
Test: 6000/10000 saved
Test: 8000/10000 saved
Test: 10000/10000 saved
✅ Fashion-MNIST resized to 64×64 using libresize.so and saved successfully!

Verification:
Shape: (64, 64, 3)
Dtype: uint8
Channels: 3
R==G: True
G==B: True
